Провести анализ датасета и построить модель для предсказания Sleep Disorder

In [ ]:
!pip install catboost

In [ ]:
# Инструменты обработки и визуализации
import pandas as pd
import numpy as np
import time
import plotly.express as px
import matplotlib.pyplot as plt

# Инструменты предобработки и разбиения
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Модели
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (VotingClassifier,
                              BaggingClassifier,
                              RandomForestClassifier,
                              GradientBoostingClassifier,
                              StackingClassifier,
                              AdaBoostClassifier)
from sklearn.tree import ExtraTreeClassifier, DecisionTreeClassifier
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB

# Метрики
from sklearn.metrics import f1_score, roc_auc_score

# Дополнительно для продвинутых техник
from sklearn.compose import TransformedTargetRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr, kruskal

# Дополнительно для многоклассовой классификации
from sklearn.multioutput import MultiOutputClassifier

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Semendyeav/datasets/refs/heads/main/PDA120_control3.csv")

# Меняем наименования колонок
df.rename(columns=lambda x: x.replace(' ','_').lower(), inplace=True)

df.head(3)

# Информация о датасете
#df.info()

# Преобразуем типы
df = df.astype({'age':'int8','quality_of_sleep':'int8','physical_activity_level':'int8','stress_level':'int8','heart_rate':'int8'})

#print(df.describe())

## Поиск и замена пропусков

Так как у нас пропуски именно в таргете - то мы не можем их заменить просто так.
Но, учитывая, что Sleep Disorder - это Расстройство сна, можно предположить, что там, где отсутствует значение - там просто не было расстройств. То есть можно заменить None на 'Without sleep disorders'

In [ ]:
# Посмотрим на пропуски
df_1 = df.copy()

# Заменим пропуска словом 'Without sleep disorders'
df['sleep_disorder'] = df['sleep_disorder'].fillna('Without sleep disorders')

# Посмотрим на пропуски
fig1 = px.imshow(df_1.isna(),title='Матрица пропусков до заполнения',width=500,height=500)
fig2 = px.imshow(df.isna(),title='Матрица пропусков после заполнения',width=500,height=500)
fig1.show()
fig2.show()

* Делаем дополнительные колонки: разделяем давления, вычилсяем разницу между ними
* Осматриваем распределения
* Преобразуем категориальные данные и пытаемся на них понять, есть ли выбросы среди других признаков (так как таргет у нас с пропусками).

## Работа с преобразованием категориальных переменных до разделения задач

In [ ]:
# Статистика по категориальным переменным
df.describe(include='object')

,gender,occupation,bmi_category,blood_pressure,sleep_disorder
count,374,374,374,374,374
unique,2,11,4,25,3
top,Male,Nurse,Normal,130/85,Without sleep disorders
freq,189,73,195,99,219


### Рассмотрим blood_pressure - кровяное давление
Разбиваем blood_pressure на три колонки:
* Систолитическое (верхнее) давление - systolic_blood_pressure - максимальная сила, с которой сердце прокачивает кровь, служит индикатором сердечной активности. Норма: 100–120 мм рт. ст.
* Диастолическое (нижнее) давление - diastolic_blood_pressure - минимальное давление в артериях и указывает на сопротивление сосудов, когда кровь не поступает. Норма: 60–80 мм рт. ст.
* Разницу между давлениями - pulse_pressure - сила, с которой сердце сокращается при каждлм ударе.Норма: 35–50 мм рт. ст.

---

### Рассмотрим bmi_category - индекс массы тела
В нормальном случае он должен делиться на:
* недостаточный вес
* норма - Normal и Normal Weight (в нашем датасете)
* избыточный вес - Overweight
* ожирение - Obese
Можно сразу перевести их в порядковые переменные, которые будут идти по повышению веса.
* 1 - Normal и Normal Weight
* 2 - Overweight
* 3 - Obese

---

### Рассмотрим gender - пол человека
Так как у нас всего два пола, то мы поступим простым образом:
* 0 - Male
* 1 - Female

---

### Рассмотрим occupation - род деятельности человека
У нас довольно мало представителей некоторых профессий:
* Manager 1
* Sales Representative	2
* Scientist 4
* Software Engineer 4
Из-за этого могут возникнуть проблемы у моделей, ибо они неправильно будут ориентироваться на этот признак. Следовательно, нужно укрупнять категории.

Объединение происходило так:
* Software Engineer & Engineer & Scientist
* Manager & Salesperson & Sales Representative

In [ ]:
#============================================================================================#
# Работа с blood_pressure
print(df['blood_pressure'].unique())
df['systolic_blood_pressure'] = df['blood_pressure'].apply(lambda x: int(x.split('/')[0]))
df['diastolic_blood_pressure'] = df['blood_pressure'].apply(lambda x: int(x.split('/')[1]))
df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']

#============================================================================================#
# Удаляем лишние колонки
print(len(df.person_id.unique()), df.shape[0]) # только уникальные значения -> удаляем
df.drop(columns=['blood_pressure','person_id'], inplace=True)

#============================================================================================#
# Работа с bmi_category
print(df.bmi_category.unique())
df['bmi_category'] = df['bmi_category'].map({'Normal':1,'Normal Weight':1,'Overweight':2,'Obese':3})

#============================================================================================#
# Работа с gender
print(df.gender.unique())
df['gender'] = df['gender'].map({'Male':0,'Female':1})

#============================================================================================#
# Работа с occupation
print(df.groupby('occupation').size())
df['occupation'] = df['occupation'].apply(lambda x: 'Engineer' if x in ('Software Engineer', 'Scientist')  else x)
df['occupation'] = df['occupation'].apply(lambda x: 'Manager' if x in ('Salesperson', 'Sales Representative') else x)
fig = px.histogram(df,x='occupation',color='sleep_disorder', barmode="group",height=500,width=1000)
#fig = px.pie(df,names='occupation',title='Распределение occupation',width=500,height=500)
fig.show()
#============================================================================================#

df.head(2)

['126/83' '125/80' '140/90' '120/80' '132/87' '130/86' '117/76' '118/76'
 '128/85' '131/86' '128/84' '115/75' '135/88' '129/84' '130/85' '115/78'
 '119/77' '121/79' '125/82' '135/90' '122/80' '142/92' '140/95' '139/91'
 '118/75']
374 374
['Overweight' 'Normal' 'Obese' 'Normal Weight']
['Male' 'Female']
occupation
Accountant              37
Doctor                  71
Engineer                63
Lawyer                  47
Manager                  1
Nurse                   73
Sales Representative     2
Salesperson             32
Scientist                4
Software Engineer        4
Teacher                 40
dtype: int64


,gender,age,occupation,sleep_duration,quality_of_sleep,physical_activity_level,stress_level,bmi_category,heart_rate,daily_steps,sleep_disorder,systolic_blood_pressure,diastolic_blood_pressure,pulse_pressure
0,0,27,Engineer,6.1,6,42,6,2,77,4200,Without sleep disorders,126,83,43
1,0,28,Doctor,6.2,6,60,8,1,75,10000,Without sleep disorders,125,80,45


In [ ]:
# Статистика по категориальным переменным
df.describe(include='object')

,occupation,sleep_disorder
count,374,374
unique,7,3
top,Nurse,Without sleep disorders
freq,73,219


## Разделение данных для разных задач

In [ ]:
# Посмотрим на распределение таргета
fig = px.pie(df,names='sleep_disorder',title='Распределение таргета',width=600,height=400)
fig.show()

In [ ]:
# Добавляем дополнительный признак таргета, чтобы попробовать декомпозировать задачу многоклассовой классификации на две задачи бинарной классификации.
df['sleep_fine'] = df['sleep_disorder'].apply(lambda x: 1 if x == 'Without sleep disorders' else 0)

# Разделим данные тестовый датасет и тренировочный
X = df.drop(columns=['sleep_disorder','sleep_fine'])
y_binary = df['sleep_fine']
y = df['sleep_disorder']

df.head(2)

,gender,age,occupation,sleep_duration,quality_of_sleep,physical_activity_level,stress_level,bmi_category,heart_rate,daily_steps,sleep_disorder,systolic_blood_pressure,diastolic_blood_pressure,pulse_pressure,sleep_fine
0,0,27,Engineer,6.1,6,42,6,2,77,4200,Without sleep disorders,126,83,43,1
1,0,28,Doctor,6.2,6,60,8,1,75,10000,Without sleep disorders,125,80,45,1


## Рассмотрим задачу бинарной классификации

In [ ]:
# 1. Списки колонок по типам обработки
num_cols = X.drop(columns=['gender','bmi_category']).select_dtypes(include=['int64', 'int8', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# 2. Трансформеры для числовых и категориальных данных
num_transformer = Pipeline(steps = [('scaler',  MinMaxScaler())])
cat_transformer = Pipeline(steps = [('encoder', TargetEncoder(target_type='continuous'))])

# 3. Собираем всё в единый препроцессор
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols),
                                               ('cat', cat_transformer, cat_cols)])

# 4. Финальный Pipeline
# Добавим несколько моделей для прогноза
# Базовые модели
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svc', svm.SVC(probability=True, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

base_models = {
    'LogisticReg':      LogisticRegression(),
    'KNeighbors':       KNeighborsClassifier(n_neighbors=7),
    'GaussianNB':       GaussianNB(),
    'SVM':              svm.SVC(kernel='rbf'),
    'DecisionTree':     DecisionTreeClassifier(criterion='gini'),
    'ExtraTree':        ExtraTreeClassifier(criterion='gini'),
    'Voting':           VotingClassifier(estimators=base_estimators,voting='soft'),
    'Bagging':          BaggingClassifier(LogisticRegression(),n_estimators=40,max_samples=100,bootstrap=True,n_jobs=-1),
    'RandomForest':     RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Stacking':         StackingClassifier(estimators=base_estimators,final_estimator=LogisticRegression(),cv=5),
    'AdaBoost':         AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),n_estimators=200,learning_rate=0.5,random_state=42),
    'XGB_clf':          XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, subsample=0.9, colsample_bytree=0.9, eval_metric="mlogloss", random_state=42,),
    'LGBM_clf':         LGBMClassifier(n_estimators=200,learning_rate=0.1, max_depth=-1,subsample=0.9, colsample_bytree=0.9, random_state=42, verbose=-1),
    'CatBoost':         CatBoostClassifier(iterations=200,learning_rate=0.1,depth=6,verbose=0,random_seed=42)
    }

base_results_bin = {}
for name, model_instance in base_models.items():
    svr_with_log = TransformedTargetRegressor(regressor=model_instance, func=np.log1p, inverse_func=np.expm1,)

    # Собираем финальные Pipeline
    final_model = Pipeline(steps=[('preprocessor', preprocessor),('model', model_instance)])

    # Разделение и обучение
    X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42)
    start = time.time()
    final_model.fit(X_train, y_train)
    time_ = time.time() - start

    # Предсказание ()
    y_pred = final_model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred)
    base_results_bin[name] = {'f1_score':f1 , 'roc_auc_score': roc, 'time': time_, 'model':final_model}

base_results_df_binary = pd.DataFrame(base_results_bin)
base_results_df_binary.T[['f1_score','roc_auc_score','time']].sort_values(by='roc_auc_score')

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



,f1_score,roc_auc_score,time
ExtraTree,0.880952,0.867733,0.014796
DecisionTree,0.894118,0.87936,0.015805
Bagging,0.921348,0.898619,6.552289
CatBoost,0.942529,0.929869,0.493783
GradientBoosting,0.954545,0.941497,0.351263
LogisticReg,0.953488,0.945494,0.045172
KNeighbors,0.953488,0.945494,0.063639
GaussianNB,0.953488,0.945494,0.039647
LGBM_clf,0.953488,0.945494,0.081239
SVM,0.965517,0.957122,0.044388


## Рассмотрим задачу многоклассовой классификации классификации

In [ ]:
l = LabelEncoder()
y_trans = l.fit_transform(y)
y_trans = pd.Series(y_trans)

In [ ]:
# 1. Списки колонок по типам обработки
num_cols = X.drop(columns=['gender','bmi_category']).select_dtypes(include=['int64', 'int8', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# 2. Трансформеры для числовых и категориальных данных
num_transformer = Pipeline(steps = [('scaler',  MinMaxScaler())])
cat_transformer = Pipeline(steps = [('encoder', TargetEncoder(target_type='continuous'))])

# 3. Собираем всё в единый препроцессор
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols),
                                               ('cat', cat_transformer, cat_cols)])

# 4. Финальный Pipeline
# Добавим несколько моделей для прогноза
# Базовые модели
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svc', svm.SVC(probability=True, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

base_models = {
    'KNeighbors':       KNeighborsClassifier(n_neighbors=7),
    'GaussianNB':       GaussianNB(),
    'SVM':              svm.SVC(kernel='rbf'),
    'DecisionTree':     DecisionTreeClassifier(criterion='gini'),
    'ExtraTree':        ExtraTreeClassifier(criterion='gini'),
    'Voting_soft':      VotingClassifier(estimators=base_estimators,voting='soft'),
    'Voting_hard':      VotingClassifier(estimators=base_estimators,voting='hard'),
    'Bagging':          BaggingClassifier(LogisticRegression(),n_estimators=40,max_samples=100,bootstrap=True,n_jobs=-1),
    'RandomForest':     RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Stacking':         StackingClassifier(estimators=base_estimators,final_estimator=LogisticRegression(),cv=5),
    'AdaBoost':         AdaBoostClassifier(DecisionTreeClassifier(max_depth=1),n_estimators=200,learning_rate=0.5,random_state=42),
    'XGB_clf':          XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, subsample=0.9, colsample_bytree=0.9, eval_metric="mlogloss", random_state=42,),
    'LGBM_clf':         LGBMClassifier(n_estimators=200,learning_rate=0.1, max_depth=-1,subsample=0.9, colsample_bytree=0.9, random_state=42, verbose=-1),
    'CatBoost':         CatBoostClassifier(iterations=200,learning_rate=0.1,depth=6,verbose=0,random_seed=42)
    }

base_results = {}
for name, model_instance in base_models.items():
    svr_with_log = TransformedTargetRegressor(regressor=model_instance, func=np.log1p, inverse_func=np.expm1,)

    # Собираем финальные Pipeline
    final_model = Pipeline(steps=[('preprocessor', preprocessor),('model', model_instance)])

    # Разделение и обучение
    X_train, X_test, y_train, y_test = train_test_split(X, y_trans, test_size=0.2, random_state=42)
    start = time.time()
    final_model.fit(X_train, y_train)
    time_ = time.time() - start

    # Предсказание ()
    y_pred = final_model.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    #roc = roc_auc_score(y_test, y_pred, multi_class='ovr')
    base_results[name] = {'f1_score':f1 , 'roc_auc_score': 0, 'time': time_, 'model':final_model}


base_results_df = pd.DataFrame(base_results)
base_results_df.T[['f1_score','roc_auc_score','time']].sort_values(by='roc_auc_score')

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



,f1_score,roc_auc_score,time
KNeighbors,0.880212,0,0.014976
GaussianNB,0.866982,0,0.01985
SVM,0.880212,0,0.028941
DecisionTree,0.822754,0,0.021624
ExtraTree,0.870013,0,0.043946
Voting_soft,0.87854,0,1.427766
Voting_hard,0.87854,0,1.825034
Bagging,0.849402,0,1.086481
RandomForest,0.866248,0,0.596499
GradientBoosting,0.84945,0,1.260609


## Попытка сделать многоклассовую классификацию вручную

In [ ]:
# 1. Списки колонок по типам обработки
num_cols = X.drop(columns=['gender','bmi_category']).select_dtypes(include=['int64', 'int8', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# 2. Трансформеры для числовых и категориальных данных
num_transformer = Pipeline(steps = [('scaler',  MinMaxScaler())])
cat_transformer = Pipeline(steps = [('encoder', TargetEncoder(target_type='continuous'))])

# 3. Собираем всё в единый препроцессор
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, num_cols),
                                               ('cat', cat_transformer, cat_cols)])

### Многоклассовая классификация вручную из двух моделей, первая из которых обучена на всех данных, а вторая - только на данных о плохом сне

In [ ]:
# Собираем Pipeline для двух моделей
first_model =  GradientBoostingClassifier(n_estimators=100, random_state=42)
second_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

final_model_first =  Pipeline(steps=[('preprocessor', preprocessor),('model', first_model)])
final_model_second = Pipeline(steps=[('preprocessor', preprocessor),('model', second_model)])

# Разделение и обучение
y_new = df['sleep_disorder']
X_train, X_test, y_train, y_test = train_test_split(X, y_new, test_size=0.2, random_state=42)
y_train_trans = y_train.apply(lambda x: 1 if x == 'Without sleep disorders' else 0)
final_model_first.fit(X_train, y_train_trans)

# Обучение второй модели на меньших признаках
df_second = df[df['sleep_fine']==0].drop(columns='sleep_fine')
df_second['sleep_disorder'] = df_second['sleep_disorder'].apply(lambda x: 2 if x == 'Sleep Apnea' else 3)
X_second = df_second.drop(columns='sleep_disorder')
y_second = df_second['sleep_disorder']
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_second, y_second, test_size=0.2, random_state=42)
final_model_second.fit(X_train_2, y_train_2)

# Предсказание ()
y_test_trans = y_test.apply(lambda x: 1 if x == 'Without sleep disorders' else 0)
y_pred = final_model_first.predict(X_test)
y_pred_second_chance = final_model_second.predict(X_test)
f1 = f1_score(y_test_trans, y_pred, average='weighted')
roc = roc_auc_score(y_test_trans, y_pred, multi_class='ovr')
df_y = pd.DataFrame()
df_y['Изначальные значения'] = y_test
df_y['Предсказанные значения первой части'] = y_pred
df_y['Предсказанные значения второй части'] = y_pred_second_chance
df_y['Попытка в предсказание'] = df_y['Предсказанные значения первой части']
df_y['Изначальные значения кодировка'] = df_y['Изначальные значения'].map({'Without sleep disorders':1, 'Sleep Apnea':2, 'Insomnia':3})
for i in df_y[df_y['Предсказанные значения первой части'] == 0].index.tolist():
    #df_y['Попытка в предсказание'].loc[[i]] = df_y['Предсказанные значения второй части'].loc[[i]]
    df_y.loc[i, 'Попытка в предсказание'] = df_y.loc[i, 'Предсказанные значения второй части']
f1 = f1_score(df_y['Изначальные значения кодировка'], df_y['Попытка в предсказание'],average='weighted')
print(f'F1-мера:{round(f1,4)}')

0.8254814814814815


### Многоклассовая классификацию вручную из трех моделей, обученных на всех данных

In [ ]:
# Собираем Pipeline для трех моделей
GB1 = GradientBoostingClassifier(n_estimators=100, random_state=42)
GB2 = GradientBoostingClassifier(n_estimators=100, random_state=42)
GB3 = GradientBoostingClassifier(n_estimators=100, random_state=42)

final_GB1 =  Pipeline(steps=[('preprocessor', preprocessor),('model', GB1)])
final_GB2 =  Pipeline(steps=[('preprocessor', preprocessor),('model', GB2)])
final_GB3 =  Pipeline(steps=[('preprocessor', preprocessor),('model', GB3)])

# Копируем датасет, добавляем новые признаки
df_triple = df.copy()

# Добавляем признаки
df_triple['sleep_fine']  = df_triple['sleep_disorder'].apply(lambda x: 1 if x == 'Without sleep disorders' else 0)
df_triple['sleep_apnea'] = df_triple['sleep_disorder'].apply(lambda x: 2 if x == 'Sleep Apnea'             else 0)
df_triple['insomnia']    = df_triple['sleep_disorder'].apply(lambda x: 3 if x == 'Insomnia'                else 0)

# Выделим признаки для алгоритмов в одном месте, чтобы не запутаться
X = df_triple.drop(columns=['sleep_disorder','sleep_fine','sleep_apnea','insomnia'])
y = df_triple[['sleep_fine','sleep_apnea','insomnia','sleep_disorder']]

# Разделение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_test['target'] = y_test['sleep_disorder'].map({'Without sleep disorders':1, 'Sleep Apnea':2, 'Insomnia':3})

# Обучение
final_GB1.fit(X_train, y_train['sleep_fine'])
final_GB2.fit(X_train, y_train['sleep_apnea'])
final_GB3.fit(X_train, y_train['insomnia'])

# Предсказание
y_GB1 = final_GB1.predict(X_test)
y_GB2 = final_GB2.predict(X_test)
y_GB3 = final_GB3.predict(X_test)

#Выведем результаты
result = pd.DataFrame({
    'sleep_fine'      :y_GB1,
    'sleep_apnea'     :y_GB2,
    'insomnia'        :y_GB3,
    'predicted_target':y_GB1
})

# Доработаем единый предсказанный таргет
# Сначала со второй моделью
for i in result[result['sleep_apnea'] != 0].index.tolist():
    if i in result[result['sleep_fine'] == 0].index.tolist():
        result.loc[i, 'predicted_target'] = result.loc[i, 'sleep_apnea']
# Потом с третьей моделью
for i in result[result['insomnia'] != 0].index.tolist():
    if i in result[result['sleep_fine'] == 0].index.tolist():
        result.loc[i, 'predicted_target'] = result.loc[i, 'insomnia']

# Посчитаем f1 метрику
f1 = f1_score(y_test['target'], result['predicted_target'],average='weighted')
print(f'F1-мера:{round(f1,4)}')

F1-мера:0.8185
